# 03 — Evaluation Traces

Evaluates the NRCS Conservation Program Navigator and produces the LangSmith traces plus the written performance commentary.

**What this notebook does**
- Runs the agent over a 10 example dataset (7 in scope, 3 out of scope) with tracing on, so every run is captured in LangSmith.
- Scores each run with a mix of deterministic evaluators and an LLM judge (a pass/fail checklist, not a 1-5 scale).
- Runs the **same dataset through two models** (premier `gpt-4o` vs. cheaper `gpt-4o-mini`) for a side by side comparison.
- Surfaces the ROI numbers (tokens, latency, cost) per model.
- Frames the human in the loop step: hand annotation in a LangSmith queue to validate the judge.

The logic lives in `src/nrcs_navigator/evaluation/` (`datasets.py`, `judge.py`, `run_traces.py`); this notebook stays thin and calls into it.

## Setup

Import the evaluation modules and confirm the model legs and LangSmith tracing are configured. `config` loads `.env` on import, so `LANGCHAIN_API_KEY` and `LANGCHAIN_TRACING_V2` are populated for the traces.

In [1]:
import asyncio
import sys

if sys.platform.startswith("win"):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

In [2]:
%load_ext autoreload
%autoreload 2

import os

from nrcs_navigator import config
from nrcs_navigator.evaluation import datasets, judge, run_traces

print("premier model:", config.PREMIER_MODEL)
print("cheap model:  ", config.CHEAP_MODEL)
print("tracing on:   ", os.environ.get("LANGCHAIN_TRACING_V2"))
print("project:      ", os.environ.get("LANGCHAIN_PROJECT"))
print("evaluators:   ", [e.__name__ for e in judge.EVALUATORS])

premier model: gpt-4o
cheap model:   gpt-4o-mini
tracing on:    true
project:       nrcs_navigator
evaluators:    ['scope_adherence', 'tool_trajectory', 'tools_succeeded', 'program_match', 'llm_judge']


## Dataset

The eval examples are defined in `datasets.py` (source of truth, in git) and pushed to a named LangSmith dataset so runs are repeatable. Each example carries `in_scope`, `expected_programs`, `expected_tools`, and a prose `expectations` rubric the judge grades against.

Push is idempotent (it recreates the dataset), so re-running this cell is safe.

In [3]:
dataset_id = datasets.push_to_langsmith()

examples = datasets.EVAL_EXAMPLES
in_scope = sum(1 for e in examples if e["in_scope"])
print(f"dataset '{datasets.DATASET_NAME}' ({dataset_id})")
print(f"{len(examples)} examples: {in_scope} in scope, {len(examples) - in_scope} out of scope")

dataset 'nrcs-navigator-eval' (bb4b6308-92fa-4b61-93f1-7a4202fb32d6)
10 examples: 6 in scope, 4 out of scope


## Run the comparison: premier vs. cheaper

`run_comparison` runs the **same dataset** through both models, applying every evaluator to each run. This is the multiple model requirement: the only thing that changes between the two legs is the agent's LLM (swapped via `model_name`), not the tools or the graph.

Runs serially because `program_availability` drives a headless browser. Each leg prints a LangSmith experiment URL — open them to inspect individual traces.

In [4]:
results = run_traces.run_comparison()
results

View the evaluation results for experiment: 'nrcs-gpt-4o-b88a8b3b' at:
https://smith.langchain.com/o/f45c547d-c2d5-437f-b620-bd76f20ae9c2/datasets/bb4b6308-92fa-4b61-93f1-7a4202fb32d6/compare?selectedSessions=95709ea1-a8fc-4109-a7e0-ba6c810e75f3




0it [00:00, ?it/s]

Screening eligibility...
Estimating payments...
Matching practices...Checking available programs...

Screening eligibility...
Estimating payments...
Matching practices...
Checking available programs...
Screening eligibility...
Checking available programs...
Estimating payments...
View the evaluation results for experiment: 'nrcs-gpt-4o-mini-e8575e47' at:
https://smith.langchain.com/o/f45c547d-c2d5-437f-b620-bd76f20ae9c2/datasets/bb4b6308-92fa-4b61-93f1-7a4202fb32d6/compare?selectedSessions=5bc940fa-7a66-4499-a04e-278e2e15556b




0it [00:00, ?it/s]

Screening eligibility...
Estimating payments...
Checking available programs...
Screening eligibility...
Estimating payments...
Matching practices...
Checking available programs...
Checking available programs...
Estimating payments...


{'gpt-4o': <ExperimentResults nrcs-gpt-4o-b88a8b3b>,
 'gpt-4o-mini': <ExperimentResults nrcs-gpt-4o-mini-e8575e47>}

## Scorecard

Mean of each feedback metric per model. Two metric families:
- **Pass/fail (binary):** `scope_adherence`, `tools_succeeded`, and the judge criteria (`no_fabrication`, `claims_cited`, `addresses_question`, `correct_redirect`) — read as pass rates. `correct_redirect` applies to out of scope examples only, so read it against the out of scope cases rather than comparing it to the in scope criteria.
- **Coverage (0-1):** `program_match`, `tool_trajectory` — read as mean coverage.

Not applicable cells (e.g. tool checks on an out of scope decline) score null and drop out of the averages.

In [5]:
scorecard = run_traces.summarize_results(results)
scorecard.round(3)

,gpt-4o,gpt-4o-mini
scope_adherence,0.889,0.778
tool_trajectory,0.792,0.611
tools_succeeded,1.000,1.000
program_match,1.000,1.000
correct_redirect,0.750,0.750
no_fabrication,0.833,0.500
claims_cited,0.500,0.167
addresses_question,0.833,0.833
avg_total_tokens,4804.800,5898.800


## ROI inputs: cost vs. effectiveness

Per model averages for latency and token usage, pulled from the experiment traces. Combined with the scorecard above, these are the raw numbers behind the cost vs. effectiveness argument: how much quality the premier model buys for its extra cost.

In [6]:
# Latency, tokens, and dollar cost per model, read from the LangSmith run roots
# (the agent invocations). Pair with the scorecard above for cost vs. effectiveness.
run_traces.roi_table(results).round(5)

,gpt-4o,gpt-4o-mini
avg_latency_s,5.41802,6.77460
avg_total_tokens,4804.80000,5898.80000
avg_cost_usd,0.01020,0.00064
total_cost_usd,0.10200,0.00643


## RAG retrieval evaluation (component eval)

Separate from the agent harness above. This tests the `eligibility_screener` retriever **directly** (`vectorstore.similarity_search`), not through the agent, so a poor score is attributable to retrieval rather than to the model's reasoning.

Each example is a question paired with the single gold CFR section that answers it. The gold citations come from the section **headings** (true structure), and the questions are deliberately **paraphrased to avoid the heading vocabulary**, so this measures semantic retrieval, not keyword overlap. Metrics:
- **hit@k** — is the gold section in the top k retrieved (recall; one gold per query).
- **MRR** — mean reciprocal rank of the gold section (rewards ranking it higher).

This is a deterministic, offline computation, so it lives in plain Python (`evaluation/rag_eval.py`) rather than LangSmith.

In [7]:
import pandas as pd

from nrcs_navigator.evaluation import rag_eval

# Summary: hit@k and MRR, unfiltered vs. with the program metadata filter.
summary = pd.concat([
    rag_eval.evaluate(use_program_filter=False),
    rag_eval.evaluate(use_program_filter=True),
])
display(summary.round(3))

# Row by row: where the gold section ranked, and the misses.
rag_eval.per_example(k=5)

,hit@1,hit@3,hit@5,MRR,n
unfiltered,0.417,0.75,0.75,0.583,12.0
filtered,0.417,0.75,0.75,0.583,12.0


,question,program,gold,rank,hit,reciprocal_rank
0,Does my client's operation qualify to take par...,CSP,7 CFR 1470.6,2.0,True,0.5
1,How is the amount of money a Conservation Stew...,CSP,7 CFR 1470.24,1.0,True,1.0
2,When joining the Conservation Stewardship Prog...,CSP,7 CFR 1470.22,2.0,True,0.5
3,After a Conservation Stewardship Program agree...,CSP,7 CFR 1470.26,1.0,True,1.0
4,"Under EQIP, how does NRCS set the dollar amoun...",EQIP,7 CFR 1466.23,2.0,True,0.5
5,What countrywide resource concerns is EQIP mea...,EQIP,7 CFR 1466.4,1.0,True,1.0
6,"When entering an EQIP contract, what document ...",EQIP,7 CFR 1466.7,2.0,True,0.5
7,"Through EQIP, how can an organization receive ...",EQIP,7 CFR 1466.32,NaN,False,0.0
8,If a farmer sells the development rights on th...,ACEP,7 CFR 1468.24,1.0,True,1.0
9,"Under ACEP, can a partner buy land, place a pe...",ACEP,7 CFR 1468.27,NaN,False,0.0


## Performance commentary

- How the agent performed overall (in scope research, out of scope declines):  
Overall, the NRCS Program Navigator performed relatively well on both in-scope and out-of-scope evaluation scenarios. For in-scope conservation planning questions, both GPT-4o and GPT-4o-mini consistently identified appropriate NRCS programs and generally maintained strong program matching performance across the evaluation set. For out-of-scope requests involving topics such as tax advice and legal interpretation, both models appropriately declined and redirected users to qualified professionals (seen in Row6-Example-#ac81 and Row7-Example-#ce53), demonstrating effective scope adherence.

- What the judge revealed:  
The judge results revealed that the primary difference between the models was not program identification accuracy but execution quality. GPT-4o was more consistent in following the intended tool trajectory and producing complete advisor-ready recommendations. In the Iowa erosion scenario (seen in Row1-Example-#0640-gpt-4o and Row1-Example-#0640-gpt-4o-mini), both models identified relevant programs such as EQIP and CSP, but GPT-4o invoked the practice-matching tool and incorporated conservation practice recommendations into its final answer, whereas GPT-4o-mini did not. In another example (seen in Row9-Example-#f7ef-gpt-4o-mini), GPT-4o-mini requested additional clarification despite already having sufficient information to evaluate eligibility, while GPT-4o proceeded directly to program guidance. These examples suggest that the largest quality differences arose from tool selection, workflow execution, and recommendation completeness rather than program identification itself.

- Premier vs. cheaper trade off read against the ROI table:  
When compared against the ROI results, GPT-4o delivered higher-quality, more advisor-ready responses but at substantially higher cost. GPT-4o averaged approximately 4,805 total tokens and 5.4 seconds latency per evaluation, compared with 5,899 tokens and 6.8 seconds for GPT-4o-mini. Total evaluation cost was roughly $0.0102 per run for GPT-4o versus $0.00064 for GPT-4o-mini, making GPT-4o approximately sixteen times more expensive. However, the trace review suggests that the additional cost primarily purchased greater reliability in tool use, answer completeness, and evidence integration rather than dramatically different program recommendations. However, the trace review suggests that the additional cost primarily purchased greater reliability in tool use, workflow execution, citation quality, and recommendation completeness rather than dramatically different program recommendations. Organizations that prioritize advisor-ready outputs and reduced review effort may find that tradeoff worthwhile despite the higher per-run cost.

- How the human was involved:  
Human involvement remained an important component of the evaluation process. Evaluation scenarios, expected outputs, and judging criteria were manually designed to reflect realistic NRCS advisory workflows. Human review of trace-level outputs was also necessary to interpret judge results correctly. For example, reviewing individual runs revealed process-level issues such as omitted tool calls (seen in Row1-Example-#0640-gpt-4o-mini), unnecessary clarification requests (seen in Row9-Example-#f7ef-gpt-4o-mini). Human review was also important for judge calibration. In Row4, GPT-4o-mini followed the system prompt literally by requesting one missing piece of information at a time, while GPT-4o gathered multiple required details in a single turn. Determining which behavior was preferable required human judegement because instruction compliance and task efficiency were not perfectly aligned. These observations informed judge calibration and helped distinguish between factual errors, process errors, and acceptable alternative responses. The resulting evaluation framework combined automated scoring with targeted human review to improve both measurement quality and future benchmark design.